# PaymentGuard: fraud classification and serving

A reproducible walkthrough of the repository's actual Python package. The bundled dataset and fraud labels are **synthetic**. Results do not establish real-world performance. GPU runtime is unnecessary.

Run the setup cell in Colab and upload `PaymentGuard.zip`. On a local Jupyter installation, skip that cell and run from the repository root. Dependency downloads require network access. A hosted Colab run was not verified during development.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile, tempfile
from google.colab import files
uploaded = files.upload()
archive = next(name for name in uploaded if name.lower().endswith(".zip"))
destination = Path(tempfile.mkdtemp(prefix="paymentguard_"))
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        target = (destination / name).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise ValueError("Unsafe archive member")
    z.extractall(destination)
repo = next(destination.rglob("pyproject.toml")).parent
os.chdir(repo)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.lock"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", "."])
print("Ready:", Path.cwd())

## 1. Generate raw transactions

The generator uses a fixed seed and noisy labels. It introduces a small late-period feature shift. Its mechanism is intentionally simple; a public-data benchmark is a separate next milestone.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from paymentguard.data import generate, add_history, load_csv, split_time
from paymentguard.train import train

Path("data").mkdir(exist_ok=True)
generate(n=30000, seed=42).to_csv("data/synthetic.csv", index=False)
raw = load_csv("data/synthetic.csv")
print("Rows:", len(raw), "Synthetic fraud rate:", raw.is_fraud.mean())
raw.head()

## 2. Inspect chronological partitions and history

History excludes the current event and simultaneous events. Earlier test events can update history without exposing test labels to fitting. Real chargebacks arrive late; this demo does not model that delay.

In [ ]:
prepared = add_history(raw)
parts = split_time(prepared)
pd.DataFrame([{"split": name, "rows": len(part), "start": part.timestamp.min(),
               "end": part.timestamp.max(), "fraud_rate": part.is_fraud.mean()}
              for name, part in zip(["train", "validation", "test"], parts)])

## 3. Train, compare, select

Fit a dummy baseline, logistic regression, and gradient boosting. Select a model and threshold using only validation labels. Freeze the policy before test evaluation. A zero-review policy is valid if tied scores prevent a feasible alternative.

In [ ]:
report = train("data/synthetic.csv", output="artifacts", seed=42, max_fpr=.01, data_kind="synthetic")
print("Selected:", report["selected_model"], "Threshold:", report["threshold"])
pd.DataFrame(report["validation"]).T[["average_precision", "recall", "false_positive_rate", "precision", "review_rate"]]

## 4. Held-out test results

Average precision summarizes ranking quality for rare positives. Recall measures captured fraud. FPR measures legitimate transactions flagged. A validation-selected 1% budget is not guaranteed on future data. Report any violation.

In [ ]:
pd.Series(report["test"])

In [ ]:
pd.DataFrame(report["test_slices"]).T

## 5. Reliability

Scores should not automatically be interpreted as calibrated probabilities. Compare observed fraud rates with mean model scores in each occupied bin; the Brier score is also recorded.

In [ ]:
pd.DataFrame(report["test_reliability"])

## 6. Inference API demonstration

TestClient runs the exact FastAPI app without opening a public port. Supply trusted past-only history summaries; this is a stateless scoring service.

In [ ]:
from fastapi.testclient import TestClient
from paymentguard.api import create_app

payload = json.loads(Path("examples/transaction.json").read_text())
with TestClient(create_app("artifacts/model.joblib")) as client:
    print("Ready:", client.get("/ready").json())
    response = client.post("/predict", json=payload)
    response.raise_for_status()
    print(json.dumps(response.json(), indent=2))

## 7. Verify the contracts

Run tests for chronology, future-feature invariance, current-event exclusion, tied thresholds, reproducibility, invalid requests, and exact offline/API parity.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

## Next experiment

Read the generated `artifacts/report.json`. Explain which model won on validation, whether test FPR met the budget, and which slices performed poorly. Then add a documented public-data adapter and label-maturity gap before making real-world performance claims. Do not choose models or tune thresholds repeatedly using the test set.